In [ ]:
import re
import pandas as pd

# ================================
# Load CSV
# ================================
base_path = "../../../../"
goto_folder = "ResultGroup/1.Wigner/"
filename = "Wigner-Baseline2-Llama.csv"
print("[INFO] Loading inference CSV...")
df = pd.read_csv(f"{base_path}{goto_folder}{filename}")

# Universal float pattern
FLOAT_PATTERN = r"(\d+(?:\.\d+)?)"

# ================================
# Extraction helper
# ================================
def extract_fields(text):
    state_match = re.search(r"(?P<state>cat|thermal|coherent|fock|random|number) state",
                            text, re.IGNORECASE)

    alpha_match   = re.search(r"(?:α|alpha)\s*(?:≈|=)\s*(\d+(?:\.\d+)?)",
                              text, re.IGNORECASE)
    density_match = re.search(r"(?:with\s*)?density(?:\s*value)?\s*(?:≈|=)\s*(\d+(?:\.\d+)?)",
                              text, re.IGNORECASE)
    photon_match  = re.search(r"(?:average\s*)?photon[s]?\s*(?:≈|=)\s*(\d+(?:\.\d+)?)",
                              text, re.IGNORECASE)

    qubits_match  = re.search(r"qubits\s*=\s*(\d+)", text)
    linear_match  = re.search(r"linear space.*?(?:from\s*)?-?(\d+)\s*to\s*-?(\d+)",
                              text, re.IGNORECASE)

    param = None
    if alpha_match:
        param = float(alpha_match.group(1))
    elif density_match:
        param = float(density_match.group(1))
    elif photon_match:
        param = float(photon_match.group(1))

    return {
        "state": state_match.group("state").lower() if state_match else None,
        "parameter": param,
        "qubits": int(qubits_match.group(1)) if qubits_match else None,
        "linear_space": abs(int(linear_match.group(2))) if linear_match else None
    }

# ================================
# Process each row
# ================================
rows = []

for idx, row in df.iterrows():
    gen = extract_fields(row["generated"])
    gt  = extract_fields(row["ground_truth"])

    rows.append({
        "test_case": row["test_case"],

        "state_generated": gen["state"],
        "param_generated": gen["parameter"],
        "qubits_generated": gen["qubits"],
        "linear_generated": gen["linear_space"],

        "state_gt": gt["state"],
        "param_gt": gt["parameter"],
        "qubits_gt": gt["qubits"],
        "linear_gt": gt["linear_space"],
    })

# ================================
# Save to CSV
# ================================
out_df = pd.DataFrame(rows)
out_df.to_csv("../2.Output/1.Converted-Wigner-Baseline2-Llama.csv", index=False)

print("Saved to ../2.Output/1.Converted-Wigner-Baseline2-Llama.csv")

[INFO] Loading inference CSV...
Saved to ../Converted-Wigner-Baseline2-Llama.csv
